In [ ]:
import re 
import json
import os
import sys
from functools import reduce
from collections import Counter, defaultdict
from typing import List, Union, Dict
import numpy as np
import ast
from copy import deepcopy
import joblib

PAI_ROOT_PATH = "../../.."
sys.path.insert(0, PAI_ROOT_PATH)

# useful link : https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

In [ ]:
def calculate_statistics(values: List[Union[float, int]]) -> Dict[str, Union[float, int]]:
    return {
        'count': len(values),
        'max': round(max(values),3),
        'mean': round(float(np.mean(values)), 3),
        'median': round(float(np.median(values)), 3),
        'min': round(min(values),3),
        'std': round(float(np.std(values)), 3),
        'sum': round(sum(values),3)
    }

In [ ]:
BENCHMARK = 'musique' # TO CHANGE
EXP_NAME = 'musique_qwen257b_weak_mixture(#8bd6e480)(v2.3.5)' # TO CHANGE
KNOWLEDGE_GRAPH_NAME = 'qwen257b_050226_v2prompts' # TO CHANGE

LOAD_QUESTION_TO_RELEVANTIDS_PATH = f"{PAI_ROOT_PATH}/../pai2_knowledge_graphs/{BENCHMARK}/{KNOWLEDGE_GRAPH_NAME}/matched_tnids_to_questions.json"

SAVE_RETRIEVED_TRIPLES_PATH = f"artifacts/{BENCHMARK}/{EXP_NAME}/retrieved_triples_info.json"
SAVE_RETRIEVED_TRIPLES_SCORES_PATH = f"artifacts/{BENCHMARK}/{EXP_NAME}/retrived_triples_scores.json"

In [ ]:
with open(LOAD_QUESTION_TO_RELEVANTIDS_PATH, 'r', encoding='utf-8') as fd:
    MATCHED_QUESTIONS_TO_RELEVANTTIDS = json.loads(fd.read())

### PAI-1

In [ ]:
LOAD_WEAK_REASEONER_KR_LOG_PATH = f"{PAI_ROOT_PATH}/../pai2_mainexps/{BENCHMARK}/{EXP_NAME}/log/qa/kg_reasoner/weak/knowledge_retriever/main/.log"
with open(LOAD_WEAK_REASEONER_KR_LOG_PATH, 'r', encoding='utf-8') as fd:
    RAW_LOG = fd.read()

In [ ]:
START_KNOWLEDGE_RETRIEVING_LINE = "START KNOWLEDGE RETRIEVING ..."
RETRIEVED_TRIPLES_LINE = "STAGE #3.2 - TRIPLETS FILTERING..."

In [ ]:
MAIN_LOG_POINTS = []
LOG_LINES = RAW_LOG.split("\n")
for idx, log_line in enumerate(LOG_LINES):
    if log_line.endswith(START_KNOWLEDGE_RETRIEVING_LINE):
        query_hash = LOG_LINES[idx+1].split(" ")[-1].strip()
        query = LOG_LINES[idx+2].split("Question: ")[1].rstrip("\n")
        MAIN_LOG_POINTS.append((idx, 'start_knowledge_retrieval', query_hash, query))
    elif log_line.endswith(RETRIEVED_TRIPLES_LINE):
        retrieved_triples_num = int(LOG_LINES[idx+1].split(" ")[-1].strip())
        triples_info = []
        for t_idx in range(retrieved_triples_num):
            cur_triplet_id = LOG_LINES[idx+2+t_idx].split(" - * [")[1].split("] ")[0]
            cur_triplet_type = LOG_LINES[idx+2+t_idx].split("type=<RelationType.")[1].split(":")[0]
            triples_info.append((cur_triplet_id, cur_triplet_type))

        MAIN_LOG_POINTS.append((idx, 'retrieved_triples', triples_info))

In [ ]:
len(MAIN_LOG_POINTS)

In [ ]:
MATCHED_TRIPLES_PER_QUESTION_INFO = dict()

cur_question_info = dict()
cur_question_hash = None 
for point_info in MAIN_LOG_POINTS:
    if point_info[1] == 'start_knowledge_retrieval':
        cur_question_hash = point_info[2]
        cur_question_info = {'question': point_info[3], 'retrieved_triples': []}

    elif point_info[1] == 'retrieved_triples':
        cur_question_info['retrieved_triples'] = point_info[2]
        MATCHED_TRIPLES_PER_QUESTION_INFO[cur_question_hash] = deepcopy(cur_question_info)

In [ ]:
# MATCHED_TRIPLES_PER_QUESTION_INFO['d8c0ae7d27b4f936ef8d795738c1dc3b']

In [ ]:
with open(SAVE_RETRIEVED_TRIPLES_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(MATCHED_TRIPLES_PER_QUESTION_INFO, ensure_ascii=False, indent=2))

In [ ]:
# calculating Precision, Recall and F1 for retrived triples that are aggregated by clue-query, search step and question

RAW_QLEVEL_SCORES = {'precision': [], 'recall': []}

for question_hash, question_info in MATCHED_TRIPLES_PER_QUESTION_INFO.items():
    full_relevant_simple_triples_amount = len(MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_triples_ids'].get('simple', []))
    full_relevant_hyper_triples_amount = len(MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_triples_ids'].get('hyper', []))

    unique_relevant_qlevel_triples = set()
    for triple_info in question_info['retrieved_triples']:
        if triple_info[0] in MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_triples_ids'].get(triple_info[1], []):
            unique_relevant_qlevel_triples.add(triple_info)
        
    qlevel_precision = len(unique_relevant_qlevel_triples) / len(question_info['retrieved_triples'])
    qlevel_recall = len(unique_relevant_qlevel_triples) / (full_relevant_simple_triples_amount + full_relevant_hyper_triples_amount)
    #elevel_f1 = (2 * elevel_precision * elevel_recall) / (elevel_precision + elevel_recall)
    RAW_QLEVEL_SCORES['precision'].append(qlevel_precision)
    RAW_QLEVEL_SCORES['recall'].append(qlevel_recall)
    #RAW_ELEVEL_SCORES['f1'].append(elevel_f1)


In [ ]:
SCORES = {group_name: {
    metric_name: calculate_statistics(metric_scores) for metric_name, metric_scores in scores_group.items()} 
    for group_name, scores_group in [('question-level', RAW_QLEVEL_SCORES)]}

In [ ]:
with open(SAVE_RETRIEVED_TRIPLES_SCORES_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(SCORES, indent=2, ensure_ascii=False))

### PAI-2

In [ ]:
LOAD_MEDIUM_REASEONER_LOG_PATH = f"{PAI_ROOT_PATH}/../pai2_mainexps/{BENCHMARK}/{EXP_NAME}/log/qa/kg_reasoner/medium/main/.log"
with open(LOAD_MEDIUM_REASEONER_LOG_PATH, 'r', encoding='utf-8') as fd:
    RAW_LOG = fd.read()

In [ ]:
START_QUERY_HANDLING_LINE = "START MEDIUM KG-REASONING..."
START_SEARCH_LINE = "STAGE#1.1 - SEARCH PLAN INITING/ENHANCING"
START_KNOWLEDGE_RETRIEVING_LINE = "STAGE#3.1.1 - KNOWLEDGE RETRIEVING"

In [ ]:
MAIN_LOG_POINTS = []
LOG_LINES = RAW_LOG.split("\n")
for idx, log_line in enumerate(LOG_LINES):
    if log_line.endswith(START_QUERY_HANDLING_LINE):
        query_hash = LOG_LINES[idx+1].split(" ")[-2]
        query = LOG_LINES[idx+2].split("Question: ")[1].rstrip("\n. ")
        MAIN_LOG_POINTS.append((idx, 'start_query_handling', query_hash, query))
    elif log_line.endswith(START_SEARCH_LINE):
        MAIN_LOG_POINTS.append((idx, 'start_search_step'))
    elif log_line.endswith(START_KNOWLEDGE_RETRIEVING_LINE):
        retrieved_triples_num = int(LOG_LINES[idx+2].split(" ")[-1].strip())
        triples_info = []
        for t_idx in range(retrieved_triples_num):
            raw_triplet_info = LOG_LINES[idx+3+t_idx].split(" - * [")[1].split("] ")[0]
            cur_triplet_id, cur_raw_relation_type = raw_triplet_info.split(" | ")
            triples_info.append((cur_triplet_id, cur_raw_relation_type.split(".")[1]))

        MAIN_LOG_POINTS.append((idx, 'knowledge_retrieving', triples_info))

In [ ]:
len(MAIN_LOG_POINTS)

In [ ]:
MATCHED_TRIPLES_PER_QUESTION_INFO = dict()

cur_question_info = dict()
cur_question_hash = None 
cur_searchstep_info = None
for point_info in MAIN_LOG_POINTS:
    if point_info[1] == 'start_query_handling':
        if cur_question_hash is not None:
            cur_question_info['search_steps'].append(deepcopy(cur_searchstep_info))
            MATCHED_TRIPLES_PER_QUESTION_INFO[cur_question_hash] = deepcopy(cur_question_info)
            cur_searchstep_info = None

        cur_question_hash = point_info[2]
        cur_question_info = {'question': point_info[3], 'search_steps': []}

    elif point_info[1] == 'start_search_step':
        if cur_searchstep_info is not None:
            cur_question_info['search_steps'].append(deepcopy(cur_searchstep_info))
        cur_searchstep_info = []

    elif point_info[1] == 'knowledge_retrieving':
        cur_searchstep_info.append(deepcopy(point_info[2]))

cur_question_info['search_steps'].append(deepcopy(cur_searchstep_info))
MATCHED_TRIPLES_PER_QUESTION_INFO[cur_question_hash] = deepcopy(cur_question_info)

In [ ]:
#MATCHED_TRIPLES_PER_QUESTION_INFO["428b91b9c26e1f2e3b7132648415012e"]

In [ ]:
#MATCHED_QUESTIONS_TO_RELEVANTTIDS["428b91b9c26e1f2e3b7132648415012e"]

In [ ]:
with open(SAVE_RETRIEVED_TRIPLES_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(MATCHED_TRIPLES_PER_QUESTION_INFO, ensure_ascii=False, indent=2))

In [ ]:
MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_triples_ids']

In [ ]:
# calculating Precision, Recall and F1 for retrived triples that are aggregated by clue-query, search step and question

RAW_QLEVEL_SCORES = {'precision': [], 'recall': []}
RAW_SLEVEL_SCORES = {'precision': [], 'recall': []}
RAW_CQLEVEL_SCORES = {'precision': [], 'recall': []}

for question_hash, question_info in MATCHED_TRIPLES_PER_QUESTION_INFO.items():
    full_relevant_simple_triples_amount = len(MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_triples_ids'].get('simple', []))
    full_relevant_hyper_triples_amount = len(MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_triples_ids'].get('hyper', []))
    
    unique_relevant_qlevel_triples = set()
    unique_irrelevant_qlevel_triples = set()
    for ss_info in question_info['search_steps']:
        unique_relevant_slevel_triples = set()
        unique_irrelevant_slevel_triples = set()

        for cq_info in ss_info:
            unique_relevant_cqlevel_triples = set()
            for triple_info in cq_info:
                if triple_info[0] in MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_triples_ids'].get(triple_info[1], []):
                    unique_relevant_cqlevel_triples.add(triple_info)
                else:
                    unique_irrelevant_slevel_triples.add(triple_info)
            
            cqlevel_precision = len(unique_relevant_cqlevel_triples) / len(cq_info)
            cqlevel_recall = len(unique_relevant_cqlevel_triples) / (full_relevant_simple_triples_amount + full_relevant_hyper_triples_amount)
            #elevel_f1 = (2 * elevel_precision * elevel_recall) / (elevel_precision + elevel_recall)
            RAW_CQLEVEL_SCORES['precision'].append(cqlevel_precision)
            RAW_CQLEVEL_SCORES['recall'].append(cqlevel_recall)
            #RAW_ELEVEL_SCORES['f1'].append(elevel_f1)

            unique_relevant_slevel_triples.update(unique_relevant_cqlevel_triples)

        slevel_precision = len(unique_relevant_slevel_triples) / (len(unique_irrelevant_slevel_triples) + len(unique_relevant_slevel_triples))
        slevel_recall = len(unique_relevant_slevel_triples) / (full_relevant_simple_triples_amount + full_relevant_hyper_triples_amount)
        #slevel_f1 = (2 * slevel_precision * slevel_recall) / (slevel_precision + slevel_recall)
        RAW_SLEVEL_SCORES['precision'].append(slevel_precision)
        RAW_SLEVEL_SCORES['recall'].append(slevel_recall)
        #RAW_SLEVEL_SCORES['f1'].append(slevel_f1)

        unique_irrelevant_qlevel_triples.update(unique_irrelevant_slevel_triples)
        unique_relevant_qlevel_triples.update(unique_relevant_slevel_triples)

    qlevel_precision = len(unique_relevant_qlevel_triples) / (len(unique_irrelevant_qlevel_triples) + len(unique_relevant_qlevel_triples))
    qlevel_recall = len(unique_relevant_qlevel_triples) / (full_relevant_simple_triples_amount + full_relevant_hyper_triples_amount)
    #qlevel_f1 = (2 * qlevel_precision * qlevel_recall) / (qlevel_precision + qlevel_recall)
    RAW_QLEVEL_SCORES['precision'].append(qlevel_precision)
    RAW_QLEVEL_SCORES['recall'].append(qlevel_recall)
    #RAW_QLEVEL_SCORES['f1'].append(qlevel_f1)

In [ ]:
SCORES = {group_name: {
    metric_name: calculate_statistics(metric_scores) for metric_name, metric_scores in scores_group.items()} 
    for group_name, scores_group in [('question-level', RAW_QLEVEL_SCORES), ('searchstep-level', RAW_SLEVEL_SCORES), ('cluequery-level', RAW_CQLEVEL_SCORES)]}

In [ ]:
with open(SAVE_RETRIEVED_TRIPLES_SCORES_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(SCORES, indent=2, ensure_ascii=False))

In [ ]:
SCORES